In [ ]:
#!pip install --force-reinstall numpy==1.26.4

In [ ]:
!pip install scikit-surprise -qq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# import các thư viên cần thiết
import pandas as pd
from surprise import SVD, Reader, Dataset, accuracy
from surprise.model_selection import train_test_split, GridSearchCV, cross_validate


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#load dataset: movie, rating
movies = pd.read_csv('/content/drive/MyDrive/Project/LAB5/ml-20m/movies.csv')
ratings = pd.read_csv('/content/drive/MyDrive/Project/LAB5/ml-20m/ratings.csv')

In [ ]:
#merge tập movie và rating theo cột movieId
df = pd.merge(ratings, movies, on='movieId')
df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,2,3.5,1112486027,Jumanji (1995),Adventure|Children|Fantasy
1,1,29,3.5,1112484676,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,1,32,3.5,1112484819,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,3.5,1112484727,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,3.5,1112484580,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [ ]:
#khởi tạo bộ dữ liệu từ dữ liệu với toàn bộ phim hiện có, neu nang qua thi 50% hoac 25% hoac so luong phu hop voi kha nang train cua ban
# movie_ids = [130219, 356, .....]
# movies = ["The Dark Knight (2011)",
#           "Cries and Whispers (Viskningar och rop) (1972)",
#           ....
#           ]

# Lấy 1% số phim ngẫu nhiên để giảm thời gian train
unique_movies = df['movieId'].unique()
sample_size = int(len(unique_movies) * 0.01)
movie_ids = pd.Series(unique_movies).sample(n=sample_size, random_state=42).tolist()
print(f"Số lượng phim được chọn: {len(movie_ids)}")

Số lượng phim được chọn: 267


In [ ]:
#tạo một DataFrame mẫu từ một DataFrame gốc
# sample_df = df[df.movieId.isin(movie_ids)]

sample_df = df[df.movieId.isin(movie_ids)]
print(f"Số lượng ratings trong sample: {len(sample_df)}")
sample_df.head()

Số lượng ratings trong sample: 181124


,userId,movieId,rating,timestamp,title,genres
99,1,3030,3.0,1112484548,Yojimbo (1961),Action|Adventure
204,2,1970,2.0,974820969,"Nightmare on Elm Street 3: Dream Warriors, A (...",Horror|Thriller
271,3,1060,5.0,944918310,Swingers (1996),Comedy|Drama
527,6,494,4.0,858275558,Executive Decision (1996),Action|Adventure|Thriller
539,6,802,3.0,858275783,Phenomenon (1996),Drama|Romance


In [ ]:
#tạo bộ dữ liệu pivot với index là userId, columns là title, values là rating

pivot_df = sample_df.pivot_table(index='userId', columns='title', values='rating')
pivot_df.head()

title,21 Jump Street (2012),3 on a Couch (Three on a Couch) (1966),Adrenalin: Fear the Rush (1996),"After Dark, My Sweet (1990)",All I Want (Try Seventeen) (2002),"All This, and Heaven Too (1940)",Amigo (2010),And Then Came Lola (2009),Archie To Riverdale and Back Again (1990),Aspen Extreme (1993),...,Whole (2003),Wild Target (2010),"Winner, The (1996)",With Great Power: The Stan Lee Story (2012),Without a Clue (1988),"Woman, The (2011)",Yankee Doodle Dandy (1942),Yellow Sky (1948),Yojimbo (1961),"Yours, Mine and Ours (2005)"
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Khỏi tạo reader với scale (1,5)

reader = Reader(rating_scale=(1, 5))

In [ ]:
#tạo data từ sample_df với reader

data = Dataset.load_from_df(sample_df[['userId', 'movieId', 'rating']], reader)

In [ ]:
# chia data thành trainset và testset theo tỉ lệ 80/20

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
#khỏi tạo model svd

svd_model = SVD()

In [ ]:
#huấn luyện model svd với hàm fit()

svd_model.fit(trainset)

In [ ]:
#dự đoán trên tập testset với hàm test()

predictions = svd_model.test(testset)

In [ ]:
#tính độ chính xác theo rmse từ module accuracy

rmse = accuracy.rmse(predictions)
print(f"RMSE trên tập test: {rmse}")

RMSE: 0.9853
RMSE trên tập test: 0.9852564596323241


In [ ]:
#chọn ra 100 dòng trong sample_df tương ứng với userId của 100 người ngẫu nhiên

random_users = sample_df['userId'].drop_duplicates().sample(n=100, random_state=42).tolist()
sample_100_df = sample_df[sample_df['userId'].isin(random_users)]
print(f"Số lượng ratings của 100 user: {len(sample_100_df)}")

Số lượng ratings của 100 user: 220


In [ ]:
# xuất ra Dự đoán cho 100 userId vừa chọn ra svd_model.predict()
# tính rmse trên danh sách 100 user này

predictions_100 = []
for _, row in sample_100_df.iterrows():
    pred = svd_model.predict(row['userId'], row['movieId'], row['rating'])
    predictions_100.append(pred)

rmse_100 = accuracy.rmse(predictions_100)
print(f"RMSE trên 100 user ngẫu nhiên: {rmse_100}")

RMSE: 0.7891
RMSE trên 100 user ngẫu nhiên: 0.7890959083940466


In [ ]:
parameter_grid = {'n_epochs': [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60],
                  'lr_all': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005],
                  'reg_all': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.001, 0.002, 0.003, 0.004, 0.005]}

In [ ]:
# Khởi tạo GridSearchCV
gs = GridSearchCV(SVD,
                  param_grid=parameter_grid,
                  measures=['rmse', 'mae'],
                  cv=3,
                  n_jobs=-1,
                  joblib_verbose=1)

In [ ]:
# huấn huyện lại gs bằng hàm fit() với data,

gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:  4.3min
[Parallel(n_jobs=-1)]: Done 446 tasks      | elapsed: 10.2min
[Parallel(n_jobs=-1)]: Done 796 tasks      | elapsed: 20.3min
[Parallel(n_jobs=-1)]: Done 1246 tasks      | elapsed: 36.0min
[Parallel(n_jobs=-1)]: Done 1796 tasks      | elapsed: 59.7min
[Parallel(n_jobs=-1)]: Done 2446 tasks      | elapsed: 93.8min
[Parallel(n_jobs=-1)]: Done 3196 tasks      | elapsed: 141.3min
[Parallel(n_jobs=-1)]: Done 3600 out of 3600 | elapsed: 170.4min finished


In [ ]:
#xuất ra giá trị tốt nhất của rmse

print(f"Best RMSE: {gs.best_score['rmse']}")

Best RMSE: 0.9875823561267953


In [ ]:
#xuất ra bộ tham số tốt nhất theo độ đo rmse

print(f"Best params (RMSE): {gs.best_params['rmse']}")

Best params (RMSE): {'n_epochs': 55, 'lr_all': 0.0004, 'reg_all': 0.005}


In [ ]:
#áp dụng bo tham số vừa tìm được tốt nhất cho svd_model

best_params = gs.best_params['rmse']
svd_model = SVD(n_epochs=best_params['n_epochs'],
                lr_all=best_params['lr_all'],
                reg_all=best_params['reg_all'])

In [ ]:
#tạo bộ dữ liệu full trainset
# trainset=data.build_full_trainset()

trainset = data.build_full_trainset()

In [ ]:
#fit model
# svd_model.fit(trainset)

svd_model.fit(trainset)

In [ ]:
# xuất ra Dự đoán lại cho 100 userId ở trên bằng model có thông số tốt nhất vừa tìm được
# tính lại rmse trên danh sách 100 user này ---> rút ra nhận xét là moddel có cải thiện kết quả không

predictions_100_best = []
for _, row in sample_100_df.iterrows():
    pred = svd_model.predict(row['userId'], row['movieId'], row['rating'])
    predictions_100_best.append(pred)

rmse_100_best = accuracy.rmse(predictions_100_best)
print(f"RMSE trên 100 user với model tối ưu: {rmse_100_best}")

# Nhận xét
print("\n--- NHẬN XÉT ---")
print(f"RMSE ban đầu (100 users): {rmse_100}")
print(f"RMSE sau tối ưu (100 users): {rmse_100_best}")
if rmse_100_best < rmse_100:
    improvement = ((rmse_100 - rmse_100_best) / rmse_100) * 100
    print(f"Model đã cải thiện {improvement:.2f}% sau khi tối ưu hyperparameters.")
else:
    print("Model không cải thiện sau khi tối ưu hyperparameters.")

RMSE: 1.0182
RMSE trên 100 user với model tối ưu: 1.0182224751820659

--- NHẬN XÉT ---
RMSE ban đầu (100 users): 0.7890959083940466
RMSE sau tối ưu (100 users): 1.0182224751820659
Model không cải thiện sau khi tối ưu hyperparameters.
